## Run inferencing on LLaVA-Med

In [ ]:
!git clone https://github.com/microsoft/LLaVA-Med.git
%cd LLaVA-Med

In [ ]:
%pip install --upgrade pip
%pip install -e .

%pip install --upgrade "transformers==4.41.2" "accelerate==0.30.1" "tokenizers<0.20"
%pip uninstall -y bitsandbytes

In [ ]:
import os
os.environ["BITSANDBYTES_NOWELCOME"] = "1"
os.environ["BITSANDBYTES_DISABLE"] = "1"
os.environ["BNB_CUDA_VERSION"] = ""
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
# !hf download --repo-type model microsoft/llava-med-v1.5-mistral-7b

In [ ]:
import os
MODEL_DIR = "/home/azureuser/.cache/huggingface/hub/models--microsoft--llava-med-v1.5-mistral-7b/snapshots"
print(os.listdir(MODEL_DIR))

MODEL_PATH = MODEL_DIR + "/91bb16c122001ddc9cf1fd36ce1dae09448943a2"
# print(MODEL_PATH)

In [ ]:
from llava.model.builder import load_pretrained_model
model_path="./.models/llava-med-v1.5-mistral-7b"
model_base=None
model_name='llava-med-v1.5-mistral-7b'
tokenizer, model, image_processor, context_len = load_pretrained_model(model_path, model_base, model_name, load_8bit=False, load_4bit=False, device="cuda")

In [ ]:
!wget https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG

In [ ]:
from PIL import Image

image = Image.open("candy.JPG").convert("RGB")
image_tensor = image_processor.preprocess(image, return_tensors="pt")["pixel_values"].to("cuda")

prompt = "What animal is on the candy?"


In [ ]:
from PIL import Image
import torch
from llava.mm_utils import process_images, tokenizer_image_token
from llava.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX
from llava.conversation import conv_templates

image = Image.open("candy.JPG").convert("RGB")

conv_mode = "mistral_instruct"
conv = conv_templates[conv_mode].copy()

prompt = "What animal is on the candy?"
inp = f"{DEFAULT_IMAGE_TOKEN}\n{prompt}"
conv.append_message(conv.roles[0], inp)
conv.append_message(conv.roles[1], None)
prompt = conv.get_prompt()

print(f"Full prompt: {prompt}")

input_ids = tokenizer_image_token(
    prompt,
    tokenizer,
    IMAGE_TOKEN_INDEX,
    return_tensors='pt'
).unsqueeze(0).to("cuda")

print(f"Input IDs shape: {input_ids.shape}")

image_tensor = process_images([image], image_processor, model)
image_tensor = image_tensor.to(dtype=torch.float16, device="cuda")

print(f"Image tensor dtype: {image_tensor.dtype}")

with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        images=image_tensor,
        image_sizes=[image.size[::-1]],
        do_sample=False,
        temperature=0,
        max_new_tokens=200,
    )

outputs = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
print("Response:", outputs)

## Generate Predictions on Visual QA Dataset

In [ ]:
from generate_predictions_vqa import run_predictions
results = run_predictions(tokenizer, model, image_processor)

In [ ]:
import pandas as pd

with open("llava_med_vqa_predictions.jsonl", 'r') as f:
    predictions = [json.loads(line) for line in f]

df = pd.DataFrame(predictions)
print(f"Total predictions: {len(df)}")
print(f"\nDomain distribution:")
print(df['domain'].apply(lambda x: [k for k, v in x.items() if v]).explode().value_counts())

print("\n" + "="*80)
print("SAMPLE PREDICTIONS")
print("="*80)
for i, row in df.head(5).iterrows():
    print(f"\n--- Question {row['question_id']} ---")
    print(f"Image: {row['image']}")
    print(f"Question: {row['question']}")
    print(f"Ground Truth: {row['ground_truth'][:200]}..." if len(row['ground_truth']) > 200 else f"Ground Truth: {row['ground_truth']}")
    print(f"Prediction: {row['prediction'][:200]}..." if len(row['prediction']) > 200 else f"Prediction: {row['prediction']}")
    print("-" * 40)

In [ ]:
df.head(10)

In [ ]:
from collections import Counter
import re

def simple_word_overlap(pred, gt):
    """Calculate word overlap between prediction and ground truth."""
    pred_words = set(re.findall(r'\w+', pred.lower()))
    gt_words = set(re.findall(r'\w+', gt.lower()))
    if not gt_words:
        return 0.0
    overlap = len(pred_words & gt_words)
    return overlap / len(gt_words)

def calculate_metrics(predictions):
    """Calculate basic evaluation metrics."""
    overlaps = []
    for pred in predictions:
        overlap = simple_word_overlap(pred['prediction'], pred['ground_truth'])
        overlaps.append(overlap)
    
    return {
        'mean_word_overlap': sum(overlaps) / len(overlaps),
        'min_word_overlap': min(overlaps),
        'max_word_overlap': max(overlaps),
        'num_samples': len(predictions)
    }

metrics = calculate_metrics(predictions)
print("Evaluation Metrics (Word Overlap with Ground Truth):")
print(f"  Mean Word Overlap: {metrics['mean_word_overlap']:.4f}")
print(f"  Min Word Overlap: {metrics['min_word_overlap']:.4f}")
print(f"  Max Word Overlap: {metrics['max_word_overlap']:.4f}")
print(f"  Number of Samples: {metrics['num_samples']}")

## Testing Tool Calling with LLaVA-Med

LLaVA-Med is not natively designed for tool calling. However, you can:
1. Use prompt engineering to get structured outputs for tool selection
2. Use a separate tool-calling model (like TxAgent) alongside LLaVA-Med
3. Fine-tune the model for tool selection (see `train_tool_selection.py` in the workspace)

In [ ]:
# Define available medical tools
MEDICAL_TOOLS = {
    "image_diagnosis": "Analyze medical image to provide diagnosis suggestions",
    "symptom_lookup": "Look up symptoms and possible conditions",
    "drug_interaction": "Check for drug interactions",
    "lab_result_analysis": "Analyze laboratory test results",
    "medical_literature": "Search medical literature for relevant information"
}

def create_tool_selection_prompt(question, tools=MEDICAL_TOOLS):
    """Create a prompt that asks the model to select appropriate tools."""
    tools_str = "\n".join([f"- {name}: {desc}" for name, desc in tools.items()])
    
    prompt = f"""You are a medical AI assistant. Given the following question, select the most appropriate tool(s) from the list below.

Available Tools:
{tools_str}

Question: {question}

Respond with ONLY the tool name(s) that should be used, separated by commas. If no tool is needed, respond with "none".
Tool selection:"""
    return prompt

# Test tool selection prompt
test_question = "What abnormalities do you see in this chest X-ray?"
tool_prompt = create_tool_selection_prompt(test_question)
print("Tool Selection Prompt:")
print(tool_prompt)

In [ ]:
def test_tool_selection_with_image(image_path, question, tokenizer, model, image_processor):
    """
    Test tool selection with LLaVA-Med by prompting it to select tools.
    Note: This is prompt-based, not native tool calling.
    """
    from PIL import Image
    import torch
    from llava.mm_utils import process_images, tokenizer_image_token
    from llava.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX
    from llava.conversation import conv_templates
    
    # Load and process image
    image = Image.open(image_path).convert("RGB")
    
    # Create tool selection prompt
    tool_prompt = create_tool_selection_prompt(question)
    
    # Format with conversation template
    conv_mode = "mistral_instruct"
    conv = conv_templates[conv_mode].copy()
    
    inp = f"{DEFAULT_IMAGE_TOKEN}\n{tool_prompt}"
    conv.append_message(conv.roles[0], inp)
    conv.append_message(conv.roles[1], None)
    full_prompt = conv.get_prompt()
    
    # Tokenize
    input_ids = tokenizer_image_token(
        full_prompt,
        tokenizer,
        IMAGE_TOKEN_INDEX,
        return_tensors='pt'
    ).unsqueeze(0).to("cuda")
    
    # Process image
    image_tensor = process_images([image], image_processor, model)
    image_tensor = image_tensor.to(dtype=torch.float16, device="cuda")
    
    # Generate
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            images=image_tensor,
            image_sizes=[image.size[::-1]],
            do_sample=False,
            temperature=0,
            max_new_tokens=50,
        )
    
    response = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
    
    # Parse tool selection from response
    # Extract the part after "Tool selection:" if present
    if "Tool selection:" in response:
        tool_selection = response.split("Tool selection:")[-1].strip()
    else:
        tool_selection = response.split("[/INST]")[-1].strip() if "[/INST]" in response else response
    
    return {
        "question": question,
        "tool_selection": tool_selection,
        "full_response": response
    }

# Example usage (uncomment after loading model):
# result = test_tool_selection_with_image("candy.JPG", "What abnormalities do you see in this image?", tokenizer, model, image_processor)
# print(f"Selected tools: {result['tool_selection']}")

In [ ]:
# Run tool selection test
result = test_tool_selection_with_image(
    "candy.JPG", 
    "What diagnosis would you suggest based on this image?", 
    tokenizer, 
    model, 
    image_processor
)

print("="*60)
print("TOOL SELECTION TEST RESULT")
print("="*60)
print(f"Question: {result['question']}")
print(f"Selected Tool(s): {result['tool_selection']}")
print(f"\nFull Response:\n{result['full_response']}")

## Option 2: Use TxAgent for Native Tool Calling

Your workspace has `run_txagent_inference.py` which uses TxAgent - a model specifically designed for tool calling in medical contexts. This is recommended for actual tool-calling functionality.

In [ ]:
# To use TxAgent for proper tool calling, run:
# python run_txagent_inference.py

# Or you can also fine-tune a model for tool selection using:
# python train_tool_selection.py

# Check existing tool selection training script
print("Available tool-calling related files in workspace:")
import os
tool_files = [f for f in os.listdir("/mnt/workspace/CorTEX") if "tool" in f.lower()]
for f in tool_files:
    print(f"  - {f}")